In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf

In [45]:
data = pd.read_csv("housing.csv")
data.dropna(inplace=True)

In [46]:
X = data.drop(columns=['median_house_value'])
X = pd.get_dummies(X, columns=['ocean_proximity'], dtype='int')
y = data['median_house_value']
X.shape

(20433, 13)

In [47]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [48]:
X_scaler = StandardScaler()
X_train_scaled = X_scaler.fit_transform(X_train)
X_val_scaled = X_scaler.transform(X_val)
X_test_scaled = X_scaler.transform(X_test)

y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(y_train.to_numpy().reshape(-1, 1))
y_val_scaled = y_scaler.transform(y_val.to_numpy().reshape(-1, 1))
y_test_scaled = y_scaler.transform(y_test.to_numpy().reshape(-1, 1))

In [49]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(13,)),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),

    tf.keras.layers.Dense(1, activation='linear')
])

In [50]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

In [51]:
model.compile(    
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae', 'mse'])

In [52]:
history = model.fit(
    X_train_scaled,
    y_train_scaled,
    epochs=200,
    validation_data = (X_val_scaled, y_val_scaled),
    batch_size = 32,
    verbose = 1,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/200
511/511 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.3747 - mae: 0.4188 - mse: 0.3747 - val_loss: 0.3579 - val_mae: 0.3818 - val_mse: 0.3579 - learning_rate: 0.0010
Epoch 2/200
511/511 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2796 - mae: 0.3630 - mse: 0.2796 - val_loss: 0.2779 - val_mae: 0.3653 - val_mse: 0.2779 - learning_rate: 0.0010
Epoch 3/200
511/511 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2553 - mae: 0.3506 - mse: 0.2553 - val_loss: 0.2660 - val_mae: 0.3582 - val_mse: 0.2660 - learning_rate: 0.0010
Epoch 4/200
511/511 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2448 - mae: 0.3410 - mse: 0.2448 - val_loss: 0.2554 - val_mae: 0.3606 - val_mse: 0.2554 - learning_rate: 0.0010
Epoch 5/200
511/511 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2373 - mae: 0.3359 - mse: 0.2373 - val_loss: 0.2522 - val_mae: 0.3480 - val_mse: 0.2522 - learning_rate: 0.0010
Epoch 6/200
511/511 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2324 - mae: 0.3306 - mse: 0.2324 - val_loss: 0.2427 - val_mae:

In [56]:
y_pred_scaled = model.predict(X_test_scaled)
y_pred = y_scaler.inverse_transform(y_pred_scaled).ravel()
y_actual = y_test.to_numpy().ravel()

# Evaluation
mae = mean_absolute_error(y_actual, y_pred)
mse = mean_squared_error(y_actual, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_actual, y_pred)

# Results
print("MAE:", mae)
print("MSE:", mse)
print("R2 Score: ",r2)
print("RMSE: ", rmse)

64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
MAE: 34103.476363747555
MSE: 2713794181.855096
R2 Score:  0.8027863745743291
RMSE:  52094.08970176076
